In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Projeto G2 — Tema 12: Mercado Imobiliário Brasileiro\n",
    "\n",
    "## 1. Introdução e Contextualização\n",
    "O mercado imobiliário possui grande importância econômica, atuando como motor no planejamento urbano e na qualidade de vida. Os preços oscilam fortemente por influência da taxa de juros, renda local e localização. A proposta deste notebook é estruturar uma análise robusta que suporte o desenvolvimento do nosso Painel Executivo (Streamlit)."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Bibliotecas e Leitura dos Dados\n",
    "Implementando `pandas` para dados, `seaborn`/`matplotlib` para gráficos e `sqlite3` para a persistência em banco."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sqlite3\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.ticker as mtick\n",
    "import seaborn as sns\n",
    "from sqlalchemy import create_engine\n",
    "\n",
    "# Configuração de Estilo Global\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "\n",
    "# Leitura\n",
    "df = pd.read_csv('dados/simulacao_mercado_imobiliario_brasil.csv')\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Limpeza e Preparação\n",
    "Ajuste da tipagem temporal e conferência de integridade da base."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df['data'] = pd.to_datetime(df['data'])\n",
    "df['ano_mes'] = df['data'].dt.to_period(\"M\").astype(str)\n",
    "\n",
    "print(\"Quantidade de Nulos:\\n\", df.isnull().sum())\n",
    "df.info()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Engenharia de Atributos e Persistência em Banco (Desafio Avançado)\n",
    "Criando faixas de tamanho para os imóveis e salvando os dados manipulados em um banco SQLite."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Engenharia\n",
    "bins = [0, 60, 120, float('inf')]\n",
    "labels = ['Pequeno (até 60m²)', 'Médio (60-120m²)', 'Grande (>120m²)']\n",
    "df['tamanho_categoria'] = pd.cut(df['area_m2'], bins=bins, labels=labels)\n",
    "\n",
    "# Banco de Dados (SQLite)\n",
    "engine = create_engine('sqlite:///mercado_imobiliario.sqlite')\n",
    "df.to_sql('imoveis', engine, if_exists='replace', index=False)\n",
    "print(\"Banco de dados atualizado com sucesso!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. KPIs e Consulta SQL Direta\n",
    "Utilizando as queries do banco para gerar os indicadores principais."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "consulta_kpi = \"\"\"\n",
    "SELECT \n",
    "    AVG(preco_imovel) AS preco_medio_nacional,\n",
    "    AVG(preco_m2) AS m2_medio,\n",
    "    AVG(taxa_juros) AS juros_medio\n",
    "FROM imoveis\n",
    "\"\"\"\n",
    "\n",
    "kpis = pd.read_sql(consulta_kpi, engine)\n",
    "print(f\"Preço Médio Nacional: R$ {kpis['preco_medio_nacional'].iloc[0]:,.2f}\")\n",
    "print(f\"Preço do M² Médio: R$ {kpis['m2_medio'].iloc[0]:,.2f}\")\n",
    "print(f\"Taxa de Juros Base: {kpis['juros_medio'].iloc[0]:.2f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Visualizações (Matplotlib & Seaborn)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 1. Evolução dos Preços\n",
    "serie_ano = df.groupby(\"ano\")[\"preco_imovel\"].mean().reset_index()\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'R$ {x/1_000:.0f}k'))\n",
    "sns.lineplot(data=serie_ano, x=\"ano\", y=\"preco_imovel\", marker=\"o\", linewidth=2, color=\"navy\", ax=ax)\n",
    "ax.set_title(\"Evolução Anual dos Preços\")\n",
    "plt.show()\n",
    "\n",
    "# 2. Preço por Nível usando a tabela do Banco\n",
    "df_sql_nivel = pd.read_sql(\"SELECT nivel_preco, AVG(preco_imovel) as preco FROM imoveis GROUP BY nivel_preco\", engine)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(8, 5))\n",
    "ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'R$ {x/1_000:.0f}k'))\n",
    "sns.barplot(data=df_sql_nivel, x=\"nivel_preco\", y=\"preco\", palette=\"magma\", ax=ax)\n",
    "ax.set_title(\"Preço Médio por Nível de Imóvel\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Interpretação e Conclusão\n",
    "A aplicação rigorosa de Pandas com SQL permitiu extrair insights poderosos. Como evidenciado, a flutuação temporal dita ciclos específicos de preço, enquanto a segmentação por nível (ex: Alto/Luxo vs. Baixo) demonstra enormes abismos monetários sustentados pela renda média regional. Estes parâmetros já foram integrados na construção do nosso App via Streamlit."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}